# Kafka 기반 광고 이벤트 Lakehouse 실습

## 실습 목표

본 실습은 Kafka로 수집된 광고 이벤트 데이터를 Spark Streaming으로 raw zone에 적재한 뒤, Apache Iceberg 기반 Silver/Gold Layer를 구성하는 것을 목표로 한다.

- Bronze/Raw: Kafka 토픽별 parquet 적재
- Silver: impression, click, conversion 이벤트를 `event_id` 기준으로 통합
- Gold: campaign 단위 광고 성과 집계
- Iceberg 기능: `MERGE INTO`, snapshot, history 확인

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-silver-gold-lab")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "/home/jovyan/warehouse")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .getOrCreate()
)

spark.sql("CREATE DATABASE IF NOT EXISTS local.ad_lakehouse")

print("SparkSession ready")

SparkSession ready


In [3]:
import os

raw_path = "/home/jovyan/warehouse/raw"

print(os.listdir(raw_path))

['impressions', 'clicks', 'conversions']


In [4]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS local.ad_lakehouse
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.ad_lakehouse.processed_events (
    event_id STRING,
    event_date DATE,
    event_time TIMESTAMP,
    impression_timestamp BIGINT,
    impression_time TIMESTAMP,
    uid STRING,
    campaign INT,
    cost DOUBLE,
    clicked INT,
    conversion INT,
    conversion_delay_sec BIGINT,
    conversion_delay_min DOUBLE,
    conversion_delay_bucket STRING,
    is_late_conversion INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (event_date)
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'copy-on-write',
    'write.merge.mode' = 'copy-on-write',
    'write.delete.mode' = 'copy-on-write',
    'write.target-file-size-bytes' = '134217728'
)
""")

print("Silver DDL completed")

Silver DDL completed


In [5]:
spark.sql("""
MERGE INTO local.ad_lakehouse.processed_events AS target
USING (
    WITH impressions_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY event_id
                    ORDER BY timestamp DESC
                ) AS rn
            FROM parquet.`/home/jovyan/warehouse/raw/impressions`
        ) t
        WHERE rn = 1
    ),
    clicks_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY event_id
                    ORDER BY timestamp DESC
                ) AS rn
            FROM parquet.`/home/jovyan/warehouse/raw/clicks`
        ) t
        WHERE rn = 1
    ),
    conversions_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY event_id
                    ORDER BY timestamp DESC
                ) AS rn
            FROM parquet.`/home/jovyan/warehouse/raw/conversions`
        ) t
        WHERE rn = 1
    )
    SELECT
        i.event_id,
        CAST(to_timestamp(i.event_time) AS DATE) AS event_date,
        to_timestamp(i.event_time) AS event_time,
        i.timestamp AS impression_timestamp,
        to_timestamp(i.event_time) AS impression_time,
        i.uid,
        i.campaign,
        i.cost,
        CASE WHEN c.event_id IS NOT NULL THEN 1 ELSE 0 END AS clicked,
        CASE WHEN v.event_id IS NOT NULL THEN 1 ELSE 0 END AS conversion,
        v.conversion_delay_sec AS conversion_delay_sec,
        CASE
            WHEN v.conversion_delay_sec IS NULL THEN NULL
            ELSE CAST(v.conversion_delay_sec AS DOUBLE) / 60.0
        END AS conversion_delay_min,
        CASE
            WHEN v.conversion_delay_sec IS NULL THEN 'no_conversion'
            WHEN v.conversion_delay_sec < 60 THEN 'under_1min'
            WHEN v.conversion_delay_sec < 300 THEN '1_to_5min'
            WHEN v.conversion_delay_sec < 1800 THEN '5_to_30min'
            WHEN v.conversion_delay_sec < 3600 THEN '30min_to_1hour'
            ELSE 'over_1hour'
        END AS conversion_delay_bucket,
        CASE
            WHEN v.conversion_delay_sec IS NOT NULL AND v.conversion_delay_sec > 0 THEN 1
            ELSE 0
        END AS is_late_conversion,
        current_timestamp() AS updated_at
    FROM impressions_dedup i
    LEFT JOIN clicks_dedup c
        ON i.event_id = c.event_id
    LEFT JOIN conversions_dedup v
        ON i.event_id = v.event_id
) AS source
ON target.event_id = source.event_id
WHEN MATCHED THEN UPDATE SET
    target.event_date = source.event_date,
    target.event_time = source.event_time,
    target.impression_timestamp = source.impression_timestamp,
    target.impression_time = source.impression_time,
    target.uid = source.uid,
    target.campaign = source.campaign,
    target.cost = source.cost,
    target.clicked = source.clicked,
    target.conversion = source.conversion,
    target.conversion_delay_sec = source.conversion_delay_sec,
    target.conversion_delay_min = source.conversion_delay_min,
    target.conversion_delay_bucket = source.conversion_delay_bucket,
    target.is_late_conversion = source.is_late_conversion,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
    event_id,
    event_date,
    event_time,
    impression_timestamp,
    impression_time,
    uid,
    campaign,
    cost,
    clicked,
    conversion,
    conversion_delay_sec,
    conversion_delay_min,
    conversion_delay_bucket,
    is_late_conversion,
    created_at,
    updated_at
)
VALUES (
    source.event_id,
    source.event_date,
    source.event_time,
    source.impression_timestamp,
    source.impression_time,
    source.uid,
    source.campaign,
    source.cost,
    source.clicked,
    source.conversion,
    source.conversion_delay_sec,
    source.conversion_delay_min,
    source.conversion_delay_bucket,
    source.is_late_conversion,
    current_timestamp(),
    source.updated_at
)
""")

print("Silver MERGE INTO completed")

Silver MERGE INTO completed


In [6]:
silver_sample = spark.sql("""
SELECT
    event_id,
    event_date,
    uid,
    campaign,
    cost,
    clicked,
    conversion,
    conversion_delay_sec,
    conversion_delay_min,
    conversion_delay_bucket,
    is_late_conversion,
    created_at,
    updated_at
FROM local.ad_lakehouse.processed_events
ORDER BY event_id
LIMIT 20
""").toPandas()

silver_sample

,event_id,event_date,uid,campaign,cost,clicked,conversion,conversion_delay_sec,conversion_delay_min,conversion_delay_bucket,is_late_conversion,created_at,updated_at
0,evt_00000000,2026-04-01,7800539,13422843,0.000027,1,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
1,evt_00000001,2026-04-01,19498757,23254639,0.000738,1,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
2,evt_00000002,2026-04-01,16984549,10341182,0.000095,1,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
3,evt_00000003,2026-04-01,19048794,3144706,0.000015,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
4,evt_00000004,2026-04-01,22773354,30801593,0.000046,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
5,evt_00000005,2026-04-01,27515737,24391301,0.000153,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
6,evt_00000006,2026-04-01,31203722,5061823,0.000036,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
7,evt_00000007,2026-04-01,29535012,828346,0.000010,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
8,evt_00000008,2026-04-01,31589113,17686799,0.000015,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328
9,evt_00000009,2026-04-01,22015892,23530192,0.000301,0,0,NaN,NaN,no_conversion,0,2026-05-01 03:31:39.892328,2026-05-01 03:31:39.892328


In [10]:
silver_snapshots = spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation,
    summary
FROM local.ad_lakehouse.processed_events.snapshots
ORDER BY committed_at DESC
LIMIT 5
""").toPandas()

silver_snapshots

,committed_at,snapshot_id,operation,summary
0,2026-05-01 03:31:42.354,8868788588001686336,overwrite,"{'spark.app.id': 'local-1777605295342', 'chang..."


In [7]:
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ad_lakehouse.campaign_summary (
    summary_date DATE,
    campaign INT,
    impressions BIGINT,
    clicks BIGINT,
    conversions BIGINT,
    total_cost DOUBLE,
    avg_conversion_delay_sec DOUBLE,
    avg_conversion_delay_min DOUBLE,
    ctr DOUBLE,
    conversion_rate DOUBLE,
    last_event_updated_at TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (summary_date)
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'copy-on-write',
    'write.merge.mode' = 'copy-on-write',
    'write.delete.mode' = 'copy-on-write',
    'write.target-file-size-bytes' = '134217728'
)
""")

print("Gold DDL completed")

Gold DDL completed


In [8]:
spark.sql("""
MERGE INTO local.ad_lakehouse.campaign_summary AS target
USING (
    SELECT
        event_date AS summary_date,
        campaign,
        COUNT(*) AS impressions,
        SUM(clicked) AS clicks,
        SUM(conversion) AS conversions,
        SUM(cost) AS total_cost,
        AVG(CASE WHEN conversion = 1 THEN conversion_delay_sec ELSE NULL END) AS avg_conversion_delay_sec,
        AVG(CASE WHEN conversion = 1 THEN conversion_delay_min ELSE NULL END) AS avg_conversion_delay_min,
        CASE
            WHEN COUNT(*) = 0 THEN 0.0
            ELSE CAST(SUM(clicked) AS DOUBLE) / CAST(COUNT(*) AS DOUBLE)
        END AS ctr,
        CASE
            WHEN SUM(clicked) = 0 THEN 0.0
            ELSE CAST(SUM(conversion) AS DOUBLE) / CAST(SUM(clicked) AS DOUBLE)
        END AS conversion_rate,
        MAX(updated_at) AS last_event_updated_at,
        current_timestamp() AS updated_at
    FROM local.ad_lakehouse.processed_events
    GROUP BY event_date, campaign
) AS source
ON target.summary_date = source.summary_date
AND target.campaign = source.campaign
WHEN MATCHED THEN UPDATE SET
    target.impressions = source.impressions,
    target.clicks = source.clicks,
    target.conversions = source.conversions,
    target.total_cost = source.total_cost,
    target.avg_conversion_delay_sec = source.avg_conversion_delay_sec,
    target.avg_conversion_delay_min = source.avg_conversion_delay_min,
    target.ctr = source.ctr,
    target.conversion_rate = source.conversion_rate,
    target.last_event_updated_at = source.last_event_updated_at,
    target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
    summary_date,
    campaign,
    impressions,
    clicks,
    conversions,
    total_cost,
    avg_conversion_delay_sec,
    avg_conversion_delay_min,
    ctr,
    conversion_rate,
    last_event_updated_at,
    created_at,
    updated_at
)
VALUES (
    source.summary_date,
    source.campaign,
    source.impressions,
    source.clicks,
    source.conversions,
    source.total_cost,
    source.avg_conversion_delay_sec,
    source.avg_conversion_delay_min,
    source.ctr,
    source.conversion_rate,
    source.last_event_updated_at,
    current_timestamp(),
    source.updated_at
)
""")

print("Gold MERGE INTO completed")

Gold MERGE INTO completed


In [9]:
gold_sample = spark.sql("""
SELECT
    summary_date,
    campaign,
    impressions,
    clicks,
    conversions,
    total_cost,
    avg_conversion_delay_sec,
    avg_conversion_delay_min,
    ctr,
    conversion_rate,
    last_event_updated_at,
    created_at,
    updated_at
FROM local.ad_lakehouse.campaign_summary
ORDER BY summary_date, campaign
LIMIT 20
""").toPandas()

gold_sample

,summary_date,campaign,impressions,clicks,conversions,total_cost,avg_conversion_delay_sec,avg_conversion_delay_min,ctr,conversion_rate,last_event_updated_at,created_at,updated_at
0,2026-04-01,73328,2,0,0,0.000084,NaN,NaN,0.000000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
1,2026-04-01,83677,2,1,0,0.002287,NaN,NaN,0.500000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
2,2026-04-01,408759,6,3,1,0.000544,1497.0,24.95,0.500000,0.333333,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
3,2026-04-01,442617,1,0,0,0.000010,NaN,NaN,0.000000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
4,2026-04-01,497590,3,0,0,0.000041,NaN,NaN,0.000000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
5,2026-04-01,497593,12,4,0,0.000245,NaN,NaN,0.333333,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
6,2026-04-01,604244,1,1,0,0.000180,NaN,NaN,1.000000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
7,2026-04-01,604249,2,0,0,0.000032,NaN,NaN,0.000000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
8,2026-04-01,690012,1,0,0,0.000010,NaN,NaN,0.000000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420
9,2026-04-01,716288,5,3,0,0.000567,NaN,NaN,0.600000,0.000000,2026-05-01 03:31:39.892328,2026-05-01 03:34:49.310420,2026-05-01 03:34:49.310420


In [11]:
gold_snapshots = spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation,
    summary
FROM local.ad_lakehouse.campaign_summary.snapshots
ORDER BY committed_at DESC
LIMIT 5
""").toPandas()

gold_snapshots

,committed_at,snapshot_id,operation,summary
0,2026-05-01 03:34:49.822,4442689051091806863,overwrite,"{'spark.app.id': 'local-1777605295342', 'chang..."


In [12]:
silver_gold_compare = spark.sql("""
WITH silver AS (
    SELECT
        COUNT(*) AS total_impressions,
        SUM(clicked) AS total_clicks,
        SUM(conversion) AS total_conversions,
        SUM(cost) AS total_cost
    FROM local.ad_lakehouse.processed_events
),
gold AS (
    SELECT
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,
        SUM(conversions) AS total_conversions,
        SUM(total_cost) AS total_cost
    FROM local.ad_lakehouse.campaign_summary
)
SELECT
    silver.total_impressions AS silver_impressions,
    gold.total_impressions AS gold_impressions,
    silver.total_clicks AS silver_clicks,
    gold.total_clicks AS gold_clicks,
    silver.total_conversions AS silver_conversions,
    gold.total_conversions AS gold_conversions,
    silver.total_cost AS silver_total_cost,
    gold.total_cost AS gold_total_cost
FROM silver
CROSS JOIN gold
""").toPandas()

silver_gold_compare

,silver_impressions,gold_impressions,silver_clicks,gold_clicks,silver_conversions,gold_conversions,silver_total_cost,gold_total_cost
0,1000,1000,350,350,47,47,0.279805,0.279805


In [ ]:
## Snapshot 검증 해석

Iceberg 테이블은 데이터 변경 작업이 발생할 때마다 snapshot을 생성한다.

Silver Layer의 `processed_events` 테이블은 `MERGE INTO` 실행 결과로 snapshot이 생성되었으며, 이는 raw 이벤트를 event_id 기준으로 통합한 결과가 Iceberg 메타데이터에 기록되었음을 의미한다.

Gold Layer의 `campaign_summary` 테이블도 `MERGE INTO` 실행 결과로 snapshot이 생성되었으며, 이는 campaign/date 단위 집계 결과가 Iceberg 테이블에 반영되었음을 의미한다.

`snapshots`와 `history` 조회 결과를 통해 각 테이블의 변경 이력과 현재 snapshot 상태를 확인할 수 있다.